# 🧠 FaceVision AI — Training on Google Colab

### 📋 Chuẩn bị trước khi chạy:
1. Tạo folder **`FaceVision`** trên Google Drive
2. Upload **2 file** vào folder đó:
   - `FaceVision_Data.zip` (114 MB — chứa 23,707 ảnh)
   - `FaceVision_Labels.csv` (1.3 MB — labels)
3. Chọn **Runtime → Change runtime type → GPU (T4)**
4. Chạy từng cell theo thứ tự ▶️

```
📁 My Drive/
└── 📁 FaceVision/
    ├── FaceVision_Data.zip      ← upload từ local
    └── FaceVision_Labels.csv    ← upload từ local
```

> ⏱️ Thời gian training: ~4-6 tiếng (T4 free) | ~2 tiếng (A100 Pro)

---
## 📦 1. Mount Drive & Cài thư viện

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Cài thư viện (Colab đã có sẵn hầu hết)
!pip install -q torch torchvision scikit-learn opencv-python-headless pillow

# Kiểm tra GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
else:
    print('⚠️ Chưa bật GPU! Vào Runtime → Change runtime type → GPU')

---
## 📁 2. Giải nén Dataset

In [ ]:
import os

# === ĐƯỜNG DẪN FILES TRÊN DRIVE ===
DRIVE_DIR = '/content/drive/MyDrive/FaceVision'
ZIP_FILE = f'{DRIVE_DIR}/FaceVision_Data.zip'
CSV_FILE = f'{DRIVE_DIR}/FaceVision_Labels.csv'

# Thư mục làm việc trên Colab (ổ SSD nhanh)
WORK_DIR = '/content/FaceVision'
IMG_DIR = f'{WORK_DIR}/data/UTKFace'
CSV_PATH = f'{WORK_DIR}/data/labels.csv'

os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{WORK_DIR}/logs', exist_ok=True)

# --- Kiểm tra file ---
print('🔍 Kiểm tra files trên Drive...')
if not os.path.exists(ZIP_FILE):
    print(f'❌ Không tìm thấy: {ZIP_FILE}')
    print(f'   → Hãy upload FaceVision_Data.zip vào My Drive/FaceVision/')
    raise FileNotFoundError('Missing FaceVision_Data.zip')

if not os.path.exists(CSV_FILE):
    print(f'❌ Không tìm thấy: {CSV_FILE}')
    print(f'   → Hãy upload FaceVision_Labels.csv vào My Drive/FaceVision/')
    raise FileNotFoundError('Missing FaceVision_Labels.csv')

print('✅ Tìm thấy cả 2 files!')

# --- Giải nén ảnh ---
print(f'\n📦 Đang giải nén FaceVision_Data.zip → {IMG_DIR}/ ...')
!unzip -q -o "{ZIP_FILE}" -d "{IMG_DIR}/"

# --- Copy labels ---
!cp "{CSV_FILE}" "{CSV_PATH}"

# --- Kiểm tra kết quả ---
img_count = len([f for f in os.listdir(IMG_DIR) if f.endswith('.jpg')])
print(f'\n✅ Hoàn tất!')
print(f'   📸 Ảnh: {img_count:,} files')
print(f'   📄 Labels: {CSV_PATH}')

if img_count < 20000:
    print(f'\n⚠️ Chỉ có {img_count} ảnh, dự kiến ~23,707. Kiểm tra lại file zip!')

---
## 🔧 3. Định nghĩa Code

In [ ]:
# ============================================================
# CONSTANTS
# ============================================================
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

GENDER_LABELS = {0: 'Male', 1: 'Female'}
RACE_LABELS = {0: 'White', 1: 'Black', 2: 'Asian', 3: 'Others'}
RACE_REMAP = {0: 0, 1: 1, 2: 2, 3: 3, 4: 3}

NUM_GENDER_CLASSES = 2
NUM_RACE_CLASSES = 4
NUM_AGE_GROUPS = 5

def age_to_group(age):
    if age <= 12: return 0
    elif age <= 19: return 1
    elif age <= 35: return 2
    elif age <= 55: return 3
    else: return 4

print('✅ Constants')

In [ ]:
# ============================================================
# PREPROCESSING (CLAHE + Gaussian Blur + Resize)
# ============================================================
import cv2
import numpy as np

def ensure_rgb(img):
    if len(img.shape) == 2: return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 4: return cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
    return img

def apply_clahe(img, clip=2.0):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8,8)).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)

def preprocess_image(img, apply_blur=False, resize=True, target_size=IMG_SIZE):
    img = ensure_rgb(img)
    img = apply_clahe(img)
    if apply_blur: img = cv2.GaussianBlur(img, (3,3), 0)
    if resize: img = cv2.resize(img, (target_size, target_size), interpolation=cv2.INTER_AREA)
    return img

print('✅ Preprocessing')

In [ ]:
# ============================================================
# MODEL — ResNet50 + SE Attention + 3 Task Heads
# ============================================================
import torch
import torch.nn as nn
from torchvision import models

class SEBlock(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        self.exc = nn.Sequential(
            nn.Linear(ch, ch//r, bias=False), nn.ReLU(True),
            nn.Linear(ch//r, ch, bias=False), nn.Sigmoid())
    def forward(self, x): return x * self.exc(x)

class FaceAttributeModel(nn.Module):
    def __init__(self, pretrained=True, age_mode='regression'):
        super().__init__()
        self.backbone = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        nf = self.backbone.fc.in_features  # 2048
        self.backbone.fc = nn.Identity()
        self.se = SEBlock(nf)
        age_out = 1 if age_mode == 'regression' else NUM_AGE_GROUPS
        self.age_head = nn.Sequential(
            nn.Linear(nf,512), nn.BatchNorm1d(512), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(512,128), nn.ReLU(True), nn.Dropout(0.2), nn.Linear(128, age_out))
        self.gender_head = nn.Sequential(
            nn.Linear(nf,256), nn.BatchNorm1d(256), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(256, NUM_GENDER_CLASSES))
        self.race_head = nn.Sequential(
            nn.Linear(nf,256), nn.BatchNorm1d(256), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(256, NUM_RACE_CLASSES))
    def forward(self, x):
        f = self.se(self.backbone(x))
        return self.age_head(f), self.gender_head(f), self.race_head(f)

# Test nhanh
_m = FaceAttributeModel(pretrained=False)
_x = torch.randn(2,3,224,224)
_a,_g,_r = _m(_x)
print(f'✅ Model OK — {sum(p.numel() for p in _m.parameters()):,} params')
del _m, _x, _a, _g, _r

In [ ]:
# ============================================================
# LOSS — FocalLoss + WingLoss + Adaptive Multi-Task
# ============================================================
import torch.nn.functional as F
import math

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, ls=0.1, cw=None):
        super().__init__()
        self.gamma, self.ls = gamma, ls
        self.register_buffer('cw', cw)
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none', label_smoothing=self.ls, weight=self.cw)
        return (((1-torch.exp(-ce))**self.gamma)*ce).mean()

class WingLoss(nn.Module):
    def __init__(self, w=10.0, eps=2.0):
        super().__init__()
        self.w, self.eps = w, eps
        self.C = w - w*math.log(1+w/eps)
    def forward(self, pred, target):
        ax = torch.clamp(torch.abs(pred-target), max=100.0)
        return torch.where(ax<self.w, self.w*torch.log1p(ax/self.eps), ax-self.C).mean()

class MultiTaskLoss(nn.Module):
    def __init__(self, adaptive=True):
        super().__init__()
        self.adaptive = adaptive
        self.age_fn = WingLoss()
        self.gender_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
        self.race_fn = FocalLoss()
        if adaptive:
            self.lv_a = nn.Parameter(torch.zeros(1))
            self.lv_g = nn.Parameter(torch.zeros(1))
            self.lv_r = nn.Parameter(torch.zeros(1))
    def forward(self, ap, gp, rp, at, gt, rt):
        la = torch.clamp(self.age_fn(ap.squeeze(), at.float()), max=100)
        lg = torch.clamp(self.gender_fn(gp, gt), max=50)
        lr = torch.clamp(self.race_fn(rp, rt), max=50)
        if self.adaptive:
            va,vg,vr = [torch.clamp(v,-4,4) for v in [self.lv_a,self.lv_g,self.lv_r]]
            total = (0.5*torch.exp(-va)*la+0.5*va + 0.5*torch.exp(-vg)*lg+0.5*vg + 0.5*torch.exp(-vr)*lr+0.5*vr)
        else:
            total = la + lg + lr
        return total, la, lg, lr

print('✅ Loss functions')

In [ ]:
# ============================================================
# DATASET & DATALOADER
# ============================================================
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import train_test_split

class UTKFaceDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir, self.transform = img_dir, transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(os.path.join(self.img_dir, row['image']))
        if img is None: img = np.zeros((IMG_SIZE,IMG_SIZE,3), dtype=np.uint8)
        else: img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = preprocess_image(img)
        img = Image.fromarray(img)
        if self.transform: img = self.transform(img)
        age = torch.tensor(int(row['age']), dtype=torch.float32)
        gender = torch.tensor(int(row['gender']), dtype=torch.long)
        race = torch.tensor(RACE_REMAP.get(int(row['race']),3), dtype=torch.long)
        return img, age, gender, race

def get_transforms(train=True, img_size=IMG_SIZE):
    if train:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(0.5), transforms.RandomRotation(15),
            transforms.RandomAffine(0, translate=(0.05,0.05), scale=(0.9,1.1)),
            transforms.ColorJitter(0.3,0.3,0.2,0.05),
            transforms.RandomPerspective(0.1, p=0.3),
            transforms.RandomGrayscale(0.05),
            transforms.GaussianBlur(3, sigma=(0.1,1.0)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            transforms.RandomErasing(0.15, scale=(0.02,0.1)),
        ])
    return transforms.Compose([
        transforms.Resize((img_size,img_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

def load_data(csv_path, img_dir, batch_size=32, num_workers=2, img_size=IMG_SIZE):
    df = pd.read_csv(csv_path)
    df['exists'] = df['image'].apply(lambda x: os.path.exists(os.path.join(img_dir, x)))
    df = df[df['exists']].drop(columns=['exists']).reset_index(drop=True)
    print(f'[Dataset] Total: {len(df)}')
    train_df, temp = train_test_split(df, test_size=0.30, random_state=42, stratify=df['gender'])
    val_df, test_df = train_test_split(temp, test_size=0.5, random_state=42, stratify=temp['gender'])
    print(f'[Dataset] Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
    train_ds = UTKFaceDataset(train_df, img_dir, get_transforms(True, img_size))
    val_ds = UTKFaceDataset(val_df, img_dir, get_transforms(False, img_size))
    test_ds = UTKFaceDataset(test_df, img_dir, get_transforms(False, img_size))
    # Weighted sampling (Race + Age)
    rr = train_df['race'].map(lambda x: RACE_REMAP.get(x,3))
    ag = train_df['age'].map(age_to_group)
    n = len(rr)
    rc = rr.value_counts().to_dict(); ac = ag.value_counts().to_dict()
    rw = {c: n/(len(rc)*v) for c,v in rc.items()}
    aw = {c: n/(len(ac)*v) for c,v in ac.items()}
    sw = np.sqrt(rr.map(rw).values.astype(np.float64) * ag.map(aw).values.astype(np.float64))
    sampler = WeightedRandomSampler(sw, len(train_df), replacement=True)
    kw = dict(num_workers=num_workers, pin_memory=True)
    return (DataLoader(train_ds, batch_size=batch_size, sampler=sampler, **kw),
            DataLoader(val_ds, batch_size=batch_size, shuffle=False, **kw),
            DataLoader(test_ds, batch_size=batch_size, shuffle=False, **kw))

# Mixup
def mixup_data(imgs, ages, genders, races, alpha=0.2):
    lam = max(np.random.beta(alpha,alpha), 0.5) if alpha>0 else 1.0
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    return lam*imgs+(1-lam)*imgs[idx], ages, ages[idx], genders, genders[idx], races, races[idx], lam

def mixup_criterion(crit, ap, gp, rp, aa, ab, ga, gb, ra, rb, lam):
    la, a1, a2, a3 = crit(ap,gp,rp,aa,ga,ra)
    lb, b1, b2, b3 = crit(ap,gp,rp,ab,gb,rb)
    return lam*la+(1-lam)*lb, lam*a1+(1-lam)*b1, lam*a2+(1-lam)*b2, lam*a3+(1-lam)*b3

print('✅ Dataset & Mixup')

---
## 🏋️ 4. Training

**2 pha training:**
- Phase 1 (Epoch 1-5): Freeze backbone, chỉ train heads → LR cao
- Phase 2 (Epoch 6+): Unfreeze backbone → differential LR (backbone chậm, heads nhanh)

**Kỹ thuật sử dụng:** Mixup, Gradient Accumulation, Progressive Resizing, Early Stopping

In [ ]:
# ============================================================
# TRAINING FUNCTIONS
# ============================================================
import time, json
from torch.amp import autocast
try:
    from torch.amp import GradScaler
except ImportError:
    from torch.cuda.amp import GradScaler
from sklearn.metrics import confusion_matrix, classification_report

def train_one_epoch(model, loader, crit, opt, device, scaler, accum=4):
    model.train()
    tl=tm=gok=rok=n=nan_c=0; opt.zero_grad()
    for step, (imgs,ages,genders,races) in enumerate(loader):
        imgs,ages,genders,races = [x.to(device) for x in [imgs,ages,genders,races]]
        with autocast(device_type='cuda'):
            if np.random.random()>0.3:
                mi,aa,ab,ga,gb,ra,rb,lam = mixup_data(imgs,ages,genders,races)
                ap,gp,rp = model(mi)
                loss,la,lg,lr = mixup_criterion(crit,ap,gp,rp,aa,ab,ga,gb,ra,rb,lam)
            else:
                ap,gp,rp = model(imgs)
                loss,la,lg,lr = crit(ap,gp,rp,ages,genders,races)
            if torch.isnan(loss) or torch.isinf(loss): nan_c+=1; opt.zero_grad(); continue
            loss = loss/accum
        scaler.scale(loss).backward()
        if (step+1)%accum==0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad()
        bs=imgs.size(0); lv=loss.item()*accum
        if not math.isnan(lv): tl+=lv*bs; tm+=la.item()*bs
        gok+=(gp.argmax(1)==genders).sum().item(); rok+=(rp.argmax(1)==races).sum().item(); n+=bs
    if nan_c: print(f'  ⚠️ {nan_c} NaN batches')
    return {'loss':tl/max(n,1),'age_mae':tm/max(n,1),'gender_acc':gok/max(n,1)*100,'race_acc':rok/max(n,1)*100}

@torch.no_grad()
def validate(model, loader, crit, device):
    model.eval(); tl=tm=gok=rok=n=0
    for imgs,ages,genders,races in loader:
        imgs,ages,genders,races = [x.to(device) for x in [imgs,ages,genders,races]]
        with autocast(device_type='cuda'):
            ap,gp,rp = model(imgs); loss,la,lg,lr = crit(ap,gp,rp,ages,genders,races)
        bs=imgs.size(0); tl+=loss.item()*bs; tm+=la.item()*bs
        gok+=(gp.argmax(1)==genders).sum().item(); rok+=(rp.argmax(1)==races).sum().item(); n+=bs
    return {'loss':tl/n,'age_mae':tm/n,'gender_acc':gok/n*100,'race_acc':rok/n*100}

@torch.no_grad()
def evaluate_test(model, loader, device):
    model.eval(); gpa,gta,rpa,rta,aea = [],[],[],[],[]
    for imgs,ages,genders,races in loader:
        with autocast(device_type='cuda'): ap,gp,rp = model(imgs.to(device))
        gpa.extend(gp.argmax(1).cpu().numpy()); gta.extend(genders.numpy())
        rpa.extend(rp.argmax(1).cpu().numpy()); rta.extend(races.numpy())
        aea.extend(torch.abs(ap.squeeze().cpu()-ages.float()).numpy())
    gn=[GENDER_LABELS[i] for i in range(2)]; rn=[RACE_LABELS[i] for i in range(4)]
    return {'age_mae':float(np.mean(aea)),
            'gender_accuracy':float(np.mean(np.array(gpa)==np.array(gta)))*100,
            'race_accuracy':float(np.mean(np.array(rpa)==np.array(rta)))*100,
            'race_macro_f1': classification_report(rta,rpa,target_names=rn,output_dict=True)['macro avg']['f1-score']*100,
            'gender_confusion_matrix':confusion_matrix(gta,gpa).tolist(),
            'race_confusion_matrix':confusion_matrix(rta,rpa).tolist(),
            'gender_classification_report':classification_report(gta,gpa,target_names=gn,output_dict=True),
            'race_classification_report':classification_report(rta,rpa,target_names=rn,output_dict=True)}

print('✅ Training functions')

In [ ]:
# ============================================================
# 🚀 BẮT ĐẦU TRAINING
# ============================================================

SAVE_PATH = f'{WORK_DIR}/checkpoints/best_model.pth'
METRICS_PATH = f'{WORK_DIR}/logs/metrics.json'
BATCH_SIZE = 32
FREEZE_EPOCHS = 5
TOTAL_EPOCHS = 50
HEAD_LR = 1e-3
BACKBONE_LR = 1e-5
PATIENCE = 10
ACCUM = 4
PROG = [(1,160),(10,192),(20,224)]  # Progressive resize

def get_sz(ep):
    s=160
    for e,sz in PROG:
        if ep>=e: s=sz
    return s

print('='*60)
print('  🧠 FACEVISION AI — TRAINING')
print('='*60)

device = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')

model = FaceAttributeModel(pretrained=True).to(device)
crit = MultiTaskLoss(adaptive=True).to(device)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

# Phase 1: Freeze backbone
for p in model.backbone.parameters(): p.requires_grad = False
for p in model.se.parameters(): p.requires_grad = True
print(f'\n⛓️ Phase 1: Freeze Backbone (Ep 1-{FREEZE_EPOCHS})')

opt = torch.optim.AdamW([
    {'params': model.se.parameters(), 'lr': HEAD_LR},
    {'params': model.age_head.parameters(), 'lr': HEAD_LR},
    {'params': model.gender_head.parameters(), 'lr': HEAD_LR},
    {'params': model.race_head.parameters(), 'lr': HEAD_LR},
    {'params': crit.parameters(), 'lr': HEAD_LR},
], weight_decay=1e-4)
scaler = GradScaler(enabled=True)

best_vl = float('inf'); no_imp = 0; hist = []; csz = 160; p2 = False
trn_ld, val_ld, tst_ld = load_data(CSV_PATH, IMG_DIR, BATCH_SIZE, 2, csz)

print(f'\n🏃 Training {TOTAL_EPOCHS} epochs...')
print('-'*60)

for ep in range(1, TOTAL_EPOCHS+1):
    t0 = time.time()

    # Phase 2
    if ep == FREEZE_EPOCHS+1 and not p2:
        print(f'\n🔓 Phase 2: Unfreeze Backbone')
        for p in model.backbone.parameters(): p.requires_grad = True
        opt = torch.optim.AdamW([
            {'params': model.backbone.parameters(), 'lr': BACKBONE_LR},
            {'params': model.se.parameters(), 'lr': HEAD_LR*0.1},
            {'params': model.age_head.parameters(), 'lr': HEAD_LR*0.1},
            {'params': model.gender_head.parameters(), 'lr': HEAD_LR*0.1},
            {'params': model.race_head.parameters(), 'lr': HEAD_LR*0.1},
            {'params': crit.parameters(), 'lr': HEAD_LR*0.1},
        ], weight_decay=1e-4)
        scaler = GradScaler(enabled=True); p2 = True

    # Progressive resize
    nsz = get_sz(ep)
    if nsz != csz:
        csz = nsz; print(f'  📐 Resize → {csz}x{csz}')
        trn_ld, val_ld, tst_ld = load_data(CSV_PATH, IMG_DIR, BATCH_SIZE, 2, csz)
        scaler = GradScaler(enabled=True)

    tm = train_one_epoch(model, trn_ld, crit, opt, device, scaler, ACCUM)
    vm = validate(model, val_ld, crit, device)
    dt = time.time()-t0

    ph = 'P1' if ep<=FREEZE_EPOCHS else 'P2'
    print(f'Ep {ep:02d}/{TOTAL_EPOCHS} ({dt:.0f}s) [{ph}] '
          f'T: L={tm["loss"]:.3f} A={tm["age_mae"]:.1f} G={tm["gender_acc"]:.1f}% R={tm["race_acc"]:.1f}% | '
          f'V: L={vm["loss"]:.3f} A={vm["age_mae"]:.1f} G={vm["gender_acc"]:.1f}% R={vm["race_acc"]:.1f}%', end='')

    hist.append({'epoch':ep, 'time':round(dt,1), 'train':tm, 'val':vm})

    if not math.isnan(vm['loss']) and vm['loss'] < best_vl:
        best_vl = vm['loss']; no_imp = 0
        torch.save({
            'epoch':ep, 'model_state_dict':model.state_dict(),
            'criterion_state_dict':crit.state_dict(),
            'val_metrics':vm, 'age_mode':'regression',
            'num_race_classes':NUM_RACE_CLASSES,
            'model_version':'resnet50_se_v3', 'img_size':csz,
        }, SAVE_PATH)
        print(f' ★ BEST')
    else:
        no_imp += 1; print(f' ({no_imp}/{PATIENCE})')
        if no_imp >= PATIENCE:
            print(f'\n🛑 Early Stopping!'); break

print(f'\n✅ Training xong! Best loss: {best_vl:.4f}')

---
## 📊 5. Đánh giá kết quả

In [ ]:
# Load best model & eval at full resolution (224x224)
if csz != 224:
    _, _, tst_ld = load_data(CSV_PATH, IMG_DIR, BATCH_SIZE, 2, 224)

ckpt = torch.load(SAVE_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
res = evaluate_test(model, tst_ld, device)

print('='*50)
print('  📊 KẾT QUẢ TEST SET')
print('='*50)
print(f'  🎂 Age MAE:         {res["age_mae"]:.2f} years')
print(f'  👤 Gender Accuracy: {res["gender_accuracy"]:.1f}%')
print(f'  🌍 Race Accuracy:   {res["race_accuracy"]:.1f}%')
print(f'  📈 Race Macro F1:   {res["race_macro_f1"]:.1f}%')
print(f'  🏆 Best Epoch:      {ckpt["epoch"]}')

print('\n  Race Per-Class:')
for name in RACE_LABELS.values():
    r = res['race_classification_report'].get(name, {})
    print(f'    {name:8s}: P={r.get("precision",0)*100:.1f}% R={r.get("recall",0)*100:.1f}% F1={r.get("f1-score",0)*100:.1f}%')

# Save
with open(METRICS_PATH, 'w') as f:
    json.dump({'best_epoch':ckpt['epoch'],'best_val_loss':best_vl,'test_metrics':res,'history':hist}, f, indent=2)
print(f'\n✅ Metrics saved!')

---
## 💾 6. Download Model về máy

**Sau khi download, copy vào project:**
```
best_model.pth → Project CV/checkpoints/best_model.pth
metrics.json   → Project CV/logs/metrics.json
```
**Rồi chạy:** `streamlit run app.py` ✅

In [ ]:
import shutil

# Backup sang Drive
drive_out = f'{DRIVE_DIR}/output'
os.makedirs(drive_out, exist_ok=True)
shutil.copy2(SAVE_PATH, f'{drive_out}/best_model.pth')
shutil.copy2(METRICS_PATH, f'{drive_out}/metrics.json')

sz = os.path.getsize(SAVE_PATH)/1024/1024
print(f'✅ Đã backup sang Drive: {drive_out}/')
print(f'   best_model.pth ({sz:.0f} MB)')
print(f'   metrics.json')

# Download trực tiếp về máy
print('\n📥 Downloading best_model.pth...')
from google.colab import files
files.download(SAVE_PATH)

In [ ]:
# Download metrics riêng
from google.colab import files
files.download(METRICS_PATH)
print('✅ Done! Copy 2 file vào project và chạy app.py')